# SMS Spam Detection — Original-only Baseline (Old Method)

This is the updated **comparison notebook** representing an original-real-SMS-only training baseline. It reads the same new `spam.csv` but filters `source == 'original'` during model fitting. It shares the *exact same original-message test split* as the new augmented model, making results comparable. It does **not** call the new augmentation training routine or save over the enhanced model.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, accuracy_score, precision_score, recall_score, f1_score
from train import DATA_PATH, load_and_split
from predict import predict_sms

if not Path('train.py').exists():
    raise RuntimeError('Please open the entire SMS_Spam_SVM_10k folder in VS Code before running this notebook.')
print('Dataset:', DATA_PATH)

## 1. Same dataset and same held-out genuine SMS

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print('Dataset shape:', df.shape)
print('Labels:', df['v1'].value_counts().to_dict())
print('\nProvenance:')
print(pd.crosstab(df['v1'], df['source']))
print('Missing values:', df.isna().sum().to_dict())
print('Exact duplicate texts:', df['v2'].duplicated().sum())
display(df[['v1', 'v2', 'source']].head(8))

In [ ]:
train_df, test_df = load_and_split()
print('Train rows:', len(train_df), ' | class counts:', train_df['v1'].value_counts().to_dict())
print('Held-out ORIGINAL test rows:', len(test_df), ' | counts:', test_df['v1'].value_counts().to_dict())
assert set(test_df['source']) == {'original'}

## 2. Fit baseline SVM without synthetic augmentation

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from train import make_pipeline

original_train = train_df.loc[train_df['source'] == 'original'].copy()
print('Original-only training rows:', len(original_train))
print(original_train['v1'].value_counts())

baseline = make_pipeline()
baseline.set_params(svm__class_weight=None)
search = GridSearchCV(baseline, {'svm__C': [0.1, 1.0, 10.0]},
                      scoring='f1_macro',
                      cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      n_jobs=1)
search.fit(original_train['v2'], original_train['v1'])
old_model = search.best_estimator_
print('Best baseline C:', search.best_params_)
print('Original-only CV macro-F1:', round(search.best_score_, 4))

## 3. Evaluate on held-out genuine SMS (compare with new notebook)

In [ ]:
y_true = test_df['v1']
y_pred = old_model.predict(test_df['v2'])
print('Accuracy:      ', round(accuracy_score(y_true, y_pred), 4))
print('Spam precision:', round(precision_score(y_true, y_pred, pos_label='spam'), 4))
print('Spam recall:   ', round(recall_score(y_true, y_pred, pos_label='spam'), 4))
print('Spam F1:       ', round(f1_score(y_true, y_pred, pos_label='spam'), 4))
print('\nOriginal-only baseline:')
print(classification_report(y_true, y_pred))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=['ham', 'spam'], cmap='Blues')
plt.title('Original-only baseline on held-out real SMS')
plt.show()

The old baseline intentionally uses the genuine-only training subset. Compare its spam recall, spam precision, and F1 with `SMS_Spam_SVM_new.ipynb` on the identical held-out messages. This notebook does not overwrite the exported augmented-model pickle.